# AI Retail Shelf Auditor (Google Colab)

**Run order:** Runtime > Change runtime type > **T4 GPU** (optional, faster) → run cells top to bottom.
Pipeline: upload shelf photo → detect products (tiled YOLO-World) → shelf rows → gaps → optional planogram check → download results.

In [ ]:
!pip install -q ultralytics
!pip install -q git+https://github.com/ultralytics/CLIP.git

In [ ]:
from pathlib import Path
from datetime import datetime
import gc, shutil

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from google.colab import files
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| OpenCV:", cv2.__version__, "| Torch:", torch.__version__)
if DEVICE == "cpu":
    print("WARNING: no GPU. The heavy model will be very slow. Runtime > Change runtime type > T4 GPU, then restart.")


## 1. Configuration (edit only this cell)

In [ ]:
PROJECT_DIR = Path("/content")
IMAGE_DIR   = PROJECT_DIR / "images"
OUTPUT_DIR  = PROJECT_DIR / "outputs"
for folder in (IMAGE_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

IMAGE_NAME = None            # None = use the image you upload in the next cell

# --- model ---
MODEL_NAME = "yolov8x-worldv2.pt"   # heavy model (needs GPU). Fallbacks: yolov8l-worldv2.pt, yolov8m-worldv2.pt
#   # open-vocabulary detector: you describe what to find in words
PROMPTS = [                          # describe the PRODUCT TYPES on your shelf
    "product box",
    "tube",
    "bottle",
    "blister pack",
]

# --- detection quality ---
TARGET_LONG_SIDE = 1920    # small photos are upscaled to this before detection
TILE_SIZE   = 800         # image is cut into overlapping tiles so small items are not missed
TILE_OVERLAP = 0.25
CONF        = 0.05        # low on purpose: dense shelves give low-confidence boxes; tune up if too noisy
NMS_IOU     = 0.50
MAX_BOX_FRACTION = 0.20   # drop boxes bigger than 20% of the image (usually a whole shelf section)

# --- audit rules ---
ROW_GAP_FACTOR  = 0.35    # new shelf row when bottom edge jumps by > 0.35 x median box height
EMPTY_GAP_FACTOR = 1.5    # gap between neighbours > 1.5 x median box width => possible empty space

## 2b. Load the uploaded photo

## 2a. Upload your shelf photo

In [ ]:
for old in IMAGE_DIR.glob("*"):      # clear previous uploads
    old.unlink()
uploaded = files.upload()            # click "Choose files" and pick your shelf photo
for name, data in uploaded.items():
    (IMAGE_DIR / name).write_bytes(data)
    IMAGE_NAME = name
print("Using:", IMAGE_NAME)

In [ ]:
candidates = sorted(p for p in IMAGE_DIR.iterdir()
                    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})

if IMAGE_NAME:
    image_path = IMAGE_DIR / IMAGE_NAME
elif candidates:
    image_path = candidates[0]
else:
    raise FileNotFoundError(f"No images found. Put a shelf photo in: {IMAGE_DIR.resolve()}")

orig_bgr = cv2.imread(str(image_path))
if orig_bgr is None:
    raise ValueError(f"Could not read image: {image_path}")

H0, W0 = orig_bgr.shape[:2]
scale = max(1.0, TARGET_LONG_SIDE / max(H0, W0))
work_bgr = (cv2.resize(orig_bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
            if scale > 1 else orig_bgr)

print(f"Image: {image_path.name} | original {W0}x{H0} | working {work_bgr.shape[1]}x{work_bgr.shape[0]} (x{scale:.2f})")
plt.figure(figsize=(8, 10)); plt.imshow(cv2.cvtColor(work_bgr, cv2.COLOR_BGR2RGB))
plt.title("Shelf photo"); plt.axis("off"); plt.show()

## 3. Load model (first run downloads weights + CLIP)

In [ ]:
model = YOLO(MODEL_NAME)
model.set_classes(PROMPTS)
print("Model ready. Looking for:", model.names)

## 4. Tiled detection

In [ ]:
def detect_tiled(model, img_bgr, tile=640, overlap=0.25, conf=0.05, iou=0.5, max_box_frac=0.2):
    """Run the model on overlapping tiles, shift boxes back, then merge duplicates with NMS."""
    h, w = img_bgr.shape[:2]
    step = max(1, int(tile * (1 - overlap)))

    def starts(size):
        if size <= tile:
            return [0]
        s = list(range(0, size - tile + 1, step))
        if s[-1] + tile < size:
            s.append(size - tile)
        return s

    boxes, scores, classes = [], [], []
    for y0 in starts(h):
        for x0 in starts(w):
            crop = img_bgr[y0:y0 + tile, x0:x0 + tile]
            res = model.predict(crop, conf=conf, imgsz=tile, device=DEVICE, verbose=False)[0]
            for b in res.boxes:
                x1, y1, x2, y2 = b.xyxy[0].tolist()
                boxes.append([x1 + x0, y1 + y0, x2 + x0, y2 + y0])
                scores.append(float(b.conf[0]))
                classes.append(int(b.cls[0]))
            del res
        gc.collect()

    if not boxes:
        return np.zeros((0, 4)), np.zeros(0), np.zeros(0, dtype=int)

    b = np.array(boxes, dtype=np.float32)
    sc = np.array(scores, dtype=np.float32)
    xywh = [[float(x1), float(y1), float(x2 - x1), float(y2 - y1)] for x1, y1, x2, y2 in b]
    keep = np.array(cv2.dnn.NMSBoxes(xywh, sc.tolist(), 0.0, iou)).reshape(-1)   # class-agnostic

    boxes, scores, classes = b[keep], sc[keep], np.array(classes)[keep]

    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    ok = areas < max_box_frac * h * w
    return boxes[ok], scores[ok], classes[ok]


boxes, scores, classes = detect_tiled(
    model, work_bgr, TILE_SIZE, TILE_OVERLAP, CONF, NMS_IOU, MAX_BOX_FRACTION
)
print("Objects detected:", len(boxes))

## 5. Detection table (coordinates in ORIGINAL image pixels)

In [ ]:
rows = []
for (x1, y1, x2, y2), s, c in zip(boxes / scale, scores, classes):
    rows.append({
        "Product_Class": model.names[int(c)],
        "Confidence": round(float(s), 3),
        "X1": int(x1), "Y1": int(y1), "X2": int(x2), "Y2": int(y2),
    })

detection_df = pd.DataFrame(rows, columns=["Product_Class", "Confidence", "X1", "Y1", "X2", "Y2"])
if not detection_df.empty:
    detection_df["Width"]  = detection_df.X2 - detection_df.X1
    detection_df["Height"] = detection_df.Y2 - detection_df.Y1

if detection_df.empty:
    summary_df = pd.DataFrame(columns=["Product_Class", "Detected_Count"])
    print("Nothing detected. Try: lower CONF, better PROMPTS, or a higher-resolution photo.")
else:
    summary_df = (detection_df.groupby("Product_Class").size()
                  .reset_index(name="Detected_Count")
                  .sort_values("Detected_Count", ascending=False)
                  .reset_index(drop=True))
display(summary_df)

## 6. Group into shelf rows and look for gaps

In [ ]:
def assign_rows(df, gap_factor):
    """Products stand on the shelf, so their BOTTOM edge (Y2) is almost constant within a row."""
    if df.empty:
        return df.assign(Shelf_Row=pd.Series(dtype=int))
    df = df.sort_values("Y2").reset_index(drop=True)
    threshold = gap_factor * df["Height"].median()
    row, prev, ids = 1, None, []
    for y2 in df["Y2"]:
        if prev is not None and (y2 - prev) > threshold:
            row += 1
        ids.append(row)
        prev = y2
    df["Shelf_Row"] = ids
    return df


def find_gaps(df, gap_factor):
    """Horizontal space between neighbours in the same row that is wider than a typical product."""
    gaps = []
    if df.empty:
        return pd.DataFrame(columns=["Shelf_Row", "X_Start", "X_End", "Y_Top", "Y_Bottom", "Gap_Width"])
    med_w = df["Width"].median()
    for row_id, grp in df.groupby("Shelf_Row"):
        grp = grp.sort_values("X1")
        y_top, y_bot = int(grp["Y1"].min()), int(grp["Y2"].max())
        right_edge = None
        for _, r in grp.iterrows():
            if right_edge is not None and (r.X1 - right_edge) > gap_factor * med_w:
                gaps.append({"Shelf_Row": row_id, "X_Start": int(right_edge), "X_End": int(r.X1),
                             "Y_Top": y_top, "Y_Bottom": y_bot, "Gap_Width": int(r.X1 - right_edge)})
            right_edge = max(right_edge, r.X2) if right_edge is not None else r.X2
    return pd.DataFrame(gaps)


detection_df = assign_rows(detection_df, ROW_GAP_FACTOR)
gaps_df = find_gaps(detection_df, EMPTY_GAP_FACTOR)

if not detection_df.empty:
    display(detection_df.groupby("Shelf_Row").size().reset_index(name="Items_In_Row"))
print("Possible empty spaces:", len(gaps_df))
display(gaps_df)

## 7. Annotated image

In [ ]:
palette = [(0, 200, 0), (255, 140, 0), (200, 0, 200), (0, 160, 255), (0, 220, 220)]
annotated = work_bgr.copy()
FONT, FONT_SCALE, THICKNESS = cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1

for (x1, y1, x2, y2), s, c in zip(boxes, scores, classes):
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    color = palette[int(c) % len(palette)]
    cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
    label = f"{model.names[int(c)]} {s:.2f}"
    (tw, th), base = cv2.getTextSize(label, FONT, FONT_SCALE, THICKNESS)
    ty = y1 if y1 - th - base >= 0 else y1 + th + base
    cv2.rectangle(annotated, (x1, ty - th - base), (x1 + tw, ty), color, -1)
    cv2.putText(annotated, label, (x1, ty - base // 2), FONT, FONT_SCALE, (255, 255, 255), THICKNESS, cv2.LINE_AA)

for _, g in gaps_df.iterrows():
    cv2.rectangle(annotated, (int(g.X_Start * scale), int(g.Y_Top * scale)),
                  (int(g.X_End * scale), int(g.Y_Bottom * scale)), (0, 0, 255), 3)

plt.figure(figsize=(12, 16)); plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f"{len(boxes)} items | {len(gaps_df)} possible gaps (red)"); plt.axis("off"); plt.show()

## 8. Stock audit (optional)
Fill `EXPECTED` with what **should** be on the shelf. Names must match `PROMPTS`. Leave it empty to skip; nothing is invented.

In [ ]:
EXPECTED = {
    # "product box": 20,
    # "tube": 10,
}

observed = detection_df["Product_Class"].value_counts() if not detection_df.empty else pd.Series(dtype=int)
audit_rows = []
for name, exp in EXPECTED.items():
    obs = int(observed.get(name, 0)); diff = exp - obs
    audit_rows.append({"Product_Class": name, "Expected_Count": exp, "Observed_Count": obs,
                       "Potential_Shortfall": max(diff, 0),
                       "Audit_Status": "Potential shortage" if diff > 0 else ("Over expected" if diff < 0 else "Matches")})
stock_audit_df = pd.DataFrame(audit_rows, columns=["Product_Class","Expected_Count","Observed_Count","Potential_Shortfall","Audit_Status"])
print("EXPECTED is empty, audit skipped." if stock_audit_df.empty else "")
display(stock_audit_df)

## 9. Save reports

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
cv2.imwrite(str(OUTPUT_DIR / f"audit_{stamp}.jpg"), annotated)
detection_df.to_csv(OUTPUT_DIR / f"detections_{stamp}.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / f"summary_{stamp}.csv", index=False)
gaps_df.to_csv(OUTPUT_DIR / f"gaps_{stamp}.csv", index=False)
stock_audit_df.to_csv(OUTPUT_DIR / f"stock_audit_{stamp}.csv", index=False)
print("Saved to:", OUTPUT_DIR.resolve())
zip_path = shutil.make_archive("/content/audit_results", "zip", OUTPUT_DIR)
files.download(zip_path)